# Demo: final model from the shipped weights → `submission.csv`

This notebook reproduces our final submission **without any training**. It loads the 20 trained fold models
(4 U-Net encoder families × 5 folds, `weights/`) and the LightGBM expected-IoU gate (`models/gate/`), predicts every
test image, and writes `submission.csv`. Requirements: `uv sync`, and the competition's test images in
`data/raw/MAGFiLO_1.0_Kaggle_2026/test/test_images/`. A GPU is recommended (~45 min on an RTX 4070; CPU works, slowly).
The stage-by-stage explanation on one training image is in `pipeline_walkthrough.ipynb`.

In [1]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
from src.config import load_config

cfg = load_config(ROOT / "configs" / "final_release.yaml", mode="infer")
test_dir = ROOT / cfg.data_dir / "test" / "test_images"
n_test = len(list(test_dir.glob("*.jpeg")))
assert n_test > 0, f"put the competition test images in {test_dir}"
print(f"device: {'cuda: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}; test images: {n_test}")

device: cuda: NVIDIA GeForce RTX 4070; test images: 180


## 1. The shipped model
Four members, each a U-Net trained with BCE + Dice semimetric loss on the annotator-mean mask (5 fold models per member,
fp16 shards below 90 MB), plus the gate: LightGBM boosters (5 seeds) that predict each candidate's expected IoU
contribution *û*; a candidate is kept when *û* > τ = 0.20 ≈ PQ/2.

In [2]:
rcfg = cfg.release
for m in rcfg.members:
    mcfg = load_config(ROOT / "configs" / f"{m}.yaml", mode="infer")
    files = sorted((ROOT / rcfg.weights_dir / m).glob("fold*.pt"))
    print(f"{m:28s} {mcfg.model.arch} + {mcfg.model.encoder:20s} {len(files):2d} files, {sum(f.stat().st_size for f in files) / 1e6:6.0f} MB")
gate = sorted((ROOT / rcfg.gate_dir).glob("lgb_seed*.txt"))
print(f"gate: {[g.name for g in gate]}, tau = {cfg.postprocess.tau}")

final_rx50_dml               Unet + resnext50_32x4d       5 files,    321 MB
final_effb4_dml              Unet + efficientnet-b4       5 files,    205 MB
final_convnext_s_dml_s43     Unet + tu-convnext_small    10 files,    536 MB
final_hrnet_w32_dml_s43      Unet + tu-hrnet_w32          5 files,    364 MB
gate: ['lgb_seed42.txt', 'lgb_seed43.txt', 'lgb_seed44.txt', 'lgb_seed45.txt', 'lgb_seed46.txt'], tau = 0.2


## 2. Predict the test set and write the submission
`src.release.run_release`: per member, the mean over its 5 fold models of the 4-way flip-TTA probability; equal-weight
mean over the members; connected components of p > 0.5 (closing radius 2, ≥ 300 px); gate; pixel-disjoint masks in
*û* order, COCO RLE. This is the same chain as training-time inference (`src/pipeline.py`).

In [3]:
from src.release import run_release
out_dir = ROOT / "outputs" / "demo"
out_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t0 = time.time()
sub = run_release(cfg, out_dir, ROOT, device, print)
print(f"done in {(time.time() - t0) / 60:.1f} min -> {out_dir / 'submission.csv'}")

C:\mydev\kaggle\fs26-fresh\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


release: final_rx50_dml fold 0 done


release: final_rx50_dml fold 1 done


release: final_rx50_dml fold 2 done


release: final_rx50_dml fold 3 done


release: final_rx50_dml fold 4 done


... (13 outputs left out)

release: final_hrnet_w32_dml_s43 fold 3 done


release: final_hrnet_w32_dml_s43 fold 4 done


release: 1445 components, keep 0.794


release: submission {'rows': 1147, 'images_with_pred': 177, 'images_missing': 3, 'per_image_mean': 6.480225988700565, 'area_median': 1311.0}
done in 29.1 min -> C:\mydev\kaggle\fs26-fresh\outputs\demo\submission.csv


## 3. Check the submission

In [4]:
from src.submission import validate_submission
stems = sorted(p.stem for p in test_dir.glob("*.jpeg"))
print(validate_submission(sub, expected_stems=stems))
per_image = sub["filament_id"].str.rsplit("_", n=1).str[0].value_counts()
print(f"{len(sub)} instances in {len(per_image)} of {len(stems)} images; per image: median {per_image.median():.0f}, max {per_image.max()}")
sub.head(3).assign(segmentation_rle=lambda d: d.segmentation_rle.str[:40] + "...")

{'rows': 1147, 'images_with_pred': 177, 'images_missing': 3, 'per_image_mean': 6.480225988700565, 'area_median': 1311.0}
1147 instances in 177 of 180 images; per image: median 6, max 18


,filament_id,segmentation_rle
0,20110120105534Ch_1,l_Vi22mo13O0O2O001O0010O01O5KOdPNDUo1<hP...
1,20110120105534Ch_2,QkTo12lo12O1O0000000O100O0010O2N10010O01...
2,20110130110334Ch_1,b_Pe2;do12N3N5K11N2OlPN@gn1`0VQNCjn1=SQN...


Optional: if a reference submission is available (e.g. the file submitted to Kaggle), compare the two instance by
instance (IoU of the masks matched per image).

In [5]:
ref_path = ROOT / "submission_reference.csv"
if ref_path.exists():
    from pycocotools import mask as mask_util
    ref = pd.read_csv(ref_path)
    obj = lambda r: {"size": [2048, 2048], "counts": r.encode()}
    ious = []
    for stem, g in sub.groupby(sub.filament_id.str.rsplit("_", n=1).str[0]):
        r = ref[ref.filament_id.str.rsplit("_", n=1).str[0] == stem]
        if len(r):
            m = np.asarray(mask_util.iou([obj(x) for x in g.segmentation_rle], [obj(x) for x in r.segmentation_rle], [0] * len(r)))
            ious += list(m.max(axis=1))
    print(f"rows {len(sub)} vs reference {len(ref)}; mean best IoU {np.mean(ious):.4f}, IoU > 0.99: {np.mean(np.array(ious) > 0.99):.3f}")
else:
    print("no submission_reference.csv: skipped")

rows 1147 vs reference 1147; mean best IoU 1.0000, IoU > 0.99: 1.000
